# 02b - Data preparation

**Aim:** prepare non-redundant positive and negative datasets for signal-peptide prediction.

We use the FASTA files from 02a to cluster proteins and retain one representative per cluster. We then recover their metadata from the collected TSV files (slides 27-29). Dataset partitioning is the next stage.

**Materials**
- `02b-DataPreparation-2026.pdf` (slides 4 and 27-29)


## 0. Load the collected datasets

We load the positive and negative FASTA files from 02a. The TSV metadata will be read after clustering.


### 0.1 Imports and paths


In [ ]:
import csv
from pathlib import Path

input_dir = Path("../data/collected")


### 0.2 Sequence reader

We keep full sequences in a dictionary keyed by `accession`.


In [2]:
def read_fasta(fasta_path):
    sequences = {}
    with open(fasta_path, encoding="utf-8") as fasta_in:
        for line in fasta_in:
            line = line.strip()
            if line.startswith(">"):
                accession = line[1:].split()[0]
                sequences[accession] = ""
            elif line:
                sequences[accession] += line
    return sequences


### 0.3 Load the collected proteins


In [3]:
positive_sequences = read_fasta(input_dir / "positive_dataset.fasta")
negative_sequences = read_fasta(input_dir / "negative_dataset.fasta")

print("Positive proteins:", len(positive_sequences))
print("Negative proteins:", len(negative_sequences))


Positive proteins: 2957
Negative proteins: 20974


## 1. Reduce redundancy

We cluster positives and negatives separately with MMseqs2, using connected components, 30% minimum identity and 40% coverage of both sequences (slides 27 and 29).

Optional installation with Conda in a Linux notebook environment. Uncomment the cell once, then comment it again for normal runs.


In [ ]:
# %conda install -c conda-forge -c bioconda mmseqs2 -y


### 1.1 Cluster the sequences

We use `easy-cluster` with the slide parameters. Each run writes a representative FASTA and a cluster-assignment TSV to `data/prepared/`.


In [ ]:
output_dir = Path("../data/prepared")
output_dir.mkdir(parents=True, exist_ok=True)

!mmseqs easy-cluster {input_dir}/positive_dataset.fasta {output_dir}/positive {output_dir}/tmp_positive --min-seq-id 0.3 -c 0.4 --cov-mode 0 --cluster-mode 1
!mmseqs easy-cluster {input_dir}/negative_dataset.fasta {output_dir}/negative {output_dir}/tmp_negative --min-seq-id 0.3 -c 0.4 --cov-mode 0 --cluster-mode 1


### 1.2 Select representatives

The `*_rep_seq.fasta` files contain one representative per cluster (slide 28).


In [ ]:
positive_representatives = read_fasta(output_dir / "positive_rep_seq.fasta")
negative_representatives = read_fasta(output_dir / "negative_rep_seq.fasta")

print("Positive representatives:", len(positive_representatives))
print("Negative representatives:", len(negative_representatives))


### 1.3 Rebuild the metadata files

We read the TSV files from 02a and retain only the representatives, preserving the original columns (slide 29). This selects rows by accession without repeating the collection filters.


In [ ]:
def read_metadata(tsv_path):
    metadata = {}
    with open(tsv_path, encoding="utf-8", newline="") as tsv_in:
        for row in csv.DictReader(tsv_in, delimiter="\t"):
            row["length"] = int(row["length"])
            if "cleavage_pos" in row:
                row["cleavage_pos"] = int(row["cleavage_pos"])
            if "has_tm_in_90" in row:
                row["has_tm_in_90"] = row["has_tm_in_90"] == "True"
            metadata[row["accession"]] = row
    return metadata



In [ ]:
positive_metadata = read_metadata(input_dir / "positive_dataset.tsv")
negative_metadata = read_metadata(input_dir / "negative_dataset.tsv")


def write_representative_metadata(metadata, representatives, tsv_path):
    columns = list(next(iter(metadata.values())))
    with open(tsv_path, "w", encoding="utf-8", newline="") as tsv_out:
        writer = csv.DictWriter(tsv_out, fieldnames=columns, delimiter="\t")
        writer.writeheader()
        for accession in representatives:
            writer.writerow(metadata[accession])


write_representative_metadata(
    positive_metadata, positive_representatives, output_dir / "positive_dataset.tsv"
)
write_representative_metadata(
    negative_metadata, negative_representatives, output_dir / "negative_dataset.tsv"
)
